In [52]:
!pip install folium pandas geopandas -q
import folium
import pandas as pd
import geopandas as gpd
from folium.plugins import HeatMap
from shapely.geometry import Polygon
from IPython.display import HTML, display

# ======================
# 1. Tạo dữ liệu điểm công cộng
# ======================
public_points = pd.DataFrame([
    {"name": "Bệnh viện A", "lat": 10.7605, "lon": 106.6650},
    {"name": "Trường học B", "lat": 10.7640, "lon": 106.6555},
    {"name": "TTTM C", "lat": 10.7588, "lon": 106.6580},
])

# ======================
# 2. Tạo dữ liệu heatmap
# ======================
transactions = pd.DataFrame([
    {"lat": 10.7630, "lon": 106.6610, "value": 120},
    {"lat": 10.7610, "lon": 106.6590, "value": 200},
    {"lat": 10.7645, "lon": 106.6640, "value": 80},
    {"lat": 10.7598, "lon": 106.6560, "value": 300},
])

# ======================
# 3. Tạo dữ liệu khu hành chính
# ======================
areas = {
    "name": ["Khu A", "Khu B"],
    "population": [12000, 18000],
    "geometry": [
        Polygon([(106.652, 10.759), (106.658, 10.759), (106.658, 10.765), (106.652, 10.765)]),
        Polygon([(106.658, 10.759), (106.664, 10.759), (106.664, 10.765), (106.658, 10.765)]),
    ]
}

gdf = gpd.GeoDataFrame(areas, crs="EPSG:4326")

# ======================
# 4. Tạo bản đồ
# ======================
m = folium.Map(location=[10.762, 106.660], zoom_start=15)

# ======================
# 5. Layer điểm công cộng
# ======================
layer_public = folium.FeatureGroup(name="Điểm công cộng")

for _, row in public_points.iterrows():
    folium.Marker(
        [row["lat"], row["lon"]],
        popup=row["name"],
        icon=folium.Icon(color="blue")
    ).add_to(layer_public)

layer_public.add_to(m)

# ======================
# 6. Layer heatmap
# ======================
layer_heat = folium.FeatureGroup(name="Heatmap giao dịch")

heat_data = transactions[["lat", "lon", "value"]].values.tolist()
HeatMap(heat_data).add_to(layer_heat)

layer_heat.add_to(m)

# ======================
# 7. Layer khu hành chính
# ======================
layer_area = folium.FeatureGroup(name="Khu hành chính")

folium.GeoJson(
    gdf,
    tooltip=folium.GeoJsonTooltip(fields=["name", "population"])
).add_to(layer_area)

layer_area.add_to(m)

# ======================
# 8. Nút bật tắt layer
# ======================
folium.LayerControl().add_to(m)

# Hiển thị bản đồ
display(HTML(m._repr_html_()))